In [2]:
import torch
assert torch.cuda.is_available(), "GPU가 없습니다. 런타임 유형을 T4 GPU로 바꾸세요."
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')
import shutil, subprocess
from pathlib import Path
subprocess.run("pip install -q ultralytics", shell=True)

SRC_DATA = Path("/content/drive/MyDrive/yolo_seg_v2_w12_e1500")
LAST = Path("/content/drive/MyDrive/seg_v2_m_1280_w12_e1500/weights/last.pt")
OUT = Path("/content/drive/MyDrive/윤수_이어학습결과")
LOCAL = Path("/content/yolo_seg_v2_w12_e1500")
print("저장본 있음?", LAST.exists(), "/ 데이터 있음?", SRC_DATA.exists())

for sub in ["images/train", "images/val", "labels/train", "labels/val"]:
    dst = LOCAL / sub; dst.mkdir(parents=True, exist_ok=True)
    for f in (SRC_DATA / sub).glob("*"):
        if not (dst / f.name).exists(): shutil.copy(f, dst / f.name)
(LOCAL / "data.yaml").write_text(f"path: {LOCAL}\ntrain: images/train\nval: images/val\n\nnames:\n  0: 도로균열\n  1: 도로홀\n", encoding="utf-8")

from ultralytics import YOLO
name = "seg_v2_m_1280_cont"
cont = OUT / name / "weights" / "last.pt"
if cont.exists():
    print("▶ 이어하기")
    YOLO(str(cont)).train(resume=True)
else:
    print("▶ 새로 시작: 9에폭")
    YOLO(str(LAST)).train(
        data=str(LOCAL / "data.yaml"), epochs=9, imgsz=1280, batch=4,
        optimizer="AdamW", lr0=0.0003, warmup_epochs=0,
        close_mosaic=6, workers=2, project=str(OUT), name=name,
    )

GPU: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
저장본 있음? True / 데이터 있음? True
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
▶ 새로 시작: 9에폭
Ultralytics 8.4.192 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=6, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_seg_v2_w12_e1500/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6